# rexgen_direct 环境配置教程

## 概述

rexgen_direct 是第一个纯图神经网络的**正向反应预测**（forward reaction prediction）方法，由 Coley 等人于 2019 年发表于 *Chemical Science*。

> **论文**: *A graph-convolutional neural network model for the prediction of chemical reactivity* (Coley et al., Chem. Sci., 2019)  
> **核心思想**: 采用两阶段流水线（two-stage pipeline），先用 Weisfeiler-Lehman Network（WLN）+ 全局注意力预测**反应中心**（哪些化学键会发生变化），再对候选产物进行枚举和排序。

### 本教程包含三份 Notebook

| 序号 | Notebook | 内容 |
|------|----------|------|
| 1 | **环境配置**（本文件） | 安装依赖、验证环境、源码结构概览 |
| 2 | 数据处理 | 反应中心提取、原子/键特征化、图构建 |
| 3 | 模型展示 | WLN、全局注意力、CoreFinder、CandRanker 的 PyTorch 实现 |

---

## 1. 核心依赖一览

rexgen_direct 原始代码基于 TensorFlow 1.x，已无法在现代环境运行（Sessions、placeholders 等 API 在 TF2 中已移除）。本教程将核心逻辑用 **PyTorch** 重写，与本项目其他教程保持一致。

| 依赖库 | 原始版本 | 教程版本（推荐） | 说明 |
|--------|----------|-----------------|------|
| Python | 2.7 / 3.6 | **3.10+** | 现代标准 |
| TensorFlow | 1.x | **PyTorch 2.0+ (CPU)** | TF1 不可安装；PyTorch 与其他教程一致 |
| RDKit | 2017.09 | **2023.09+** | 化学信息学工具包 |
| NumPy | 1.12 | **1.24+** | 数值计算 |

> **不需要**: DGL、DGL-Life、torch-geometric。rexgen_direct 使用自定义邻接表图结构，不依赖任何图框架。

## 2. 创建环境并安装依赖

推荐使用 `setup_env.sh` 脚本一键创建 Conda 环境。也可以在已有环境中手动安装。

In [1]:
import os
from pathlib import Path

def find_project_root(start=None):
    """向上查找项目根目录（包含 teaching_demos/ 的目录）"""
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
ENV_DIR = PROJECT_ROOT / 'envs' / 'rexgen_direct_tutorial_envs'
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'rexgen_direct'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'虚拟环境目录: {ENV_DIR}')
print(f'源码目录: {SOURCE_REPO}')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
虚拟环境目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/rexgen_direct_tutorial_envs
源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/rexgen_direct


In [2]:
%%bash -s "$ENV_DIR"

ENV_DIR=$1

source "$(conda info --base)/etc/profile.d/conda.sh"

if [ ! -d "$ENV_DIR" ]; then
    echo "==> 正在创建 Conda 环境 $ENV_DIR ..."
    conda create -y -p "$ENV_DIR" python=3.11
    echo "==> Conda 环境创建完成"
else
    echo "==> Conda 环境已存在: $ENV_DIR"
fi

conda activate "$ENV_DIR"

echo "==> 安装核心依赖 ..."
conda install -y -c conda-forge numpy pandas tqdm rdkit matplotlib ipykernel

echo "==> 安装 PyTorch（CPU 版）..."
pip install -q torch --index-url https://download.pytorch.org/whl/cpu

echo "==> 所有依赖安装完成"

==> 正在创建 Conda 环境 /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/rexgen_direct_tutorial_envs ...
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/rexgen_direct_tutorial_envs

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    python-3.11.15             |       h741d88c_0        29.5 MB  defaults
    ------------------------------------------------------------
                                           Total:        29.5 MB

The following NEW packages will be INSTALLED:

  _libgcc_mutex      anaconda/pkgs/main/linux-64::_libgcc_mutex-0.1-main 
  _openmp_mutex      anaconda/pkgs/main/linux-64::_openmp_mutex-5.1-1_gnu 
  bzip2              anaconda/pk



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/rexgen_direct_tutorial_envs

  added / updated specs:
    - ipykernel
    - matplotlib
    - numpy
    - pandas
    - rdkit
    - tqdm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    harfbuzz-13.1.0            |       h6083320_0         2.5 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         2.5 MB

The following NEW packages will be INSTALLED:

  alsa-lib           conda-forge/linux-64::alsa-lib-1.2.15.3-hb03c661_0 
  asttokens          conda-forge/noarch::asttokens-3.0.1-pyhd8ed1ab_0 
  brotli             conda-forge/linux-64::brotli-1.2.0-hed03a55_1 
  brotli-bin         conda-forge/linux-64::brotli-bin-1.2.0-hb03c661_1 
  cairo              conda-forge/linux-64::cairo

### 注册 Jupyter Kernel（可选）

In [3]:
%%bash -s "$ENV_DIR"
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate "$1"
python -m ipykernel install --user --name rexgen_direct_tutorial --display-name "rexgen_direct Tutorial"
echo "==> Kernel 注册完成，请在 Jupyter 中选择 'rexgen_direct Tutorial' kernel"

Installed kernelspec rexgen_direct_tutorial in /home/xiaoruiwang/.local/share/jupyter/kernels/rexgen_direct_tutorial
==> Kernel 注册完成，请在 Jupyter 中选择 'rexgen_direct Tutorial' kernel


## 3. 验证环境

运行以下代码验证所有核心依赖是否安装成功。

In [2]:
import sys
print(f'Python 版本: {sys.version}')
print(f'Python 路径: {sys.executable}')
print()

checks = {}

try:
    import torch
    checks['PyTorch'] = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError as e:
    checks['PyTorch'] = f'FAIL: {e}'

try:
    from rdkit import Chem, rdBase
    checks['RDKit'] = rdBase.rdkitVersion
except ImportError as e:
    checks['RDKit'] = f'FAIL: {e}'

try:
    import numpy as np
    checks['NumPy'] = np.__version__
except ImportError as e:
    checks['NumPy'] = f'FAIL: {e}'

try:
    import pandas as pd
    checks['pandas'] = pd.__version__
except ImportError as e:
    checks['pandas'] = f'FAIL: {e}'

try:
    import matplotlib
    checks['matplotlib'] = matplotlib.__version__
except ImportError as e:
    checks['matplotlib'] = f'FAIL: {e}'

print('=' * 50)
print('环境检查结果')
print('=' * 50)
all_ok = True
for name, status in checks.items():
    flag = '✓' if 'FAIL' not in str(status) else '✗'
    if 'FAIL' in str(status):
        all_ok = False
    print(f'  {flag} {name:20s} {status}')

print('=' * 50)
if all_ok:
    print('所有依赖检查通过！')
else:
    print('部分依赖缺失，请检查上方安装步骤。')

Python 版本: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Python 路径: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/rexgen_direct_tutorial_envs/bin/python

环境检查结果
  ✓ PyTorch              2.10.0+cpu (CUDA: False)
  ✓ RDKit                2025.09.6
  ✓ NumPy                2.4.2
  ✓ pandas               3.0.1
  ✓ matplotlib           3.10.8
所有依赖检查通过！


## 4. 源码目录结构

rexgen_direct 的代码分为两个阶段（Stage），每个阶段有独立的模型和数据处理逻辑。

In [3]:
structure = """
rexgen_direct 源码结构
══════════════════════════════════════════════════════════════
rexgen_direct/
├── core_wln_global/               # Stage 1: 反应中心预测
│   ├── mol_graph.py               #   分子图特征化（atom_features, bond_features, smiles2graph）
│   ├── nn.py                      #   linearND 层定义（TF1 实现）
│   ├── models.py                  #   WLN 消息传递网络（rcnn_wl_last）
│   ├── ioutils_direct.py          #   二值特征 + 键变化标签
│   ├── directcorefinder.py        #   CoreFinder 完整模型（全局注意力 + 预测）
│   ├── nntrain_direct.py          #   训练脚本
│   └── nntest_direct.py           #   测试脚本
│
├── rank_diff_wln/                 # Stage 2: 候选产物排序
│   ├── mol_graph_direct_useScores.py  #   候选产物图特征化 + 枚举逻辑
│   ├── nn.py                      #   linearND 层定义（同 Stage 1）
│   ├── models.py                  #   差异 WLN 网络（wl_diff_net）
│   ├── edit_mol_direct_useScores.py   #   分子编辑（应用键变化生成产物）
│   ├── directcandranker.py        #   CandRanker 完整模型
│   ├── nntrain_direct_useScores.py    #   训练脚本
│   └── nntest_direct_useScores.py     #   测试脚本
│
├── scripts/                       # 工具脚本
│   ├── prep_data.py               #   数据预处理（get_changed_bonds）
│   └── eval_by_smiles.py          #   评估脚本（edit_mol + SMILES 比较）
│
└── data/                          # 数据目录
    ├── train.txt.proc             #   训练数据（反应SMILES + 键变化）
    ├── valid.txt.proc             #   验证数据
    └── test.txt.proc              #   测试数据
"""
print(structure)


rexgen_direct 源码结构
══════════════════════════════════════════════════════════════
rexgen_direct/
├── core_wln_global/               # Stage 1: 反应中心预测
│   ├── mol_graph.py               #   分子图特征化（atom_features, bond_features, smiles2graph）
│   ├── nn.py                      #   linearND 层定义（TF1 实现）
│   ├── models.py                  #   WLN 消息传递网络（rcnn_wl_last）
│   ├── ioutils_direct.py          #   二值特征 + 键变化标签
│   ├── directcorefinder.py        #   CoreFinder 完整模型（全局注意力 + 预测）
│   ├── nntrain_direct.py          #   训练脚本
│   └── nntest_direct.py           #   测试脚本
│
├── rank_diff_wln/                 # Stage 2: 候选产物排序
│   ├── mol_graph_direct_useScores.py  #   候选产物图特征化 + 枚举逻辑
│   ├── nn.py                      #   linearND 层定义（同 Stage 1）
│   ├── models.py                  #   差异 WLN 网络（wl_diff_net）
│   ├── edit_mol_direct_useScores.py   #   分子编辑（应用键变化生成产物）
│   ├── directcandranker.py        #   CandRanker 完整模型
│   ├── nntrain_direct_useScores.py    #   训练脚本
│   └── nntest_direct_useSc

### 关键文件职责表

| 文件 | 阶段 | 职责 | 教学版改动 |
|------|------|------|------------|
| `mol_graph.py` (core) | Stage 1 | 原子特征(82维)、键特征(6维)、图构建 | **直接复用**（纯 RDKit/NumPy） |
| `ioutils_direct.py` | Stage 1 | 二值关系特征(10维)、键变化标签 | **直接复用** |
| `nn.py` | 通用 | linearND 多维线性层 | **PyTorch 重写** |
| `models.py` (core) | Stage 1 | WLN 消息传递 (rcnn_wl_last) | **PyTorch 重写** |
| `directcorefinder.py` | Stage 1 | 全局注意力 + CoreFinder | **PyTorch 重写** |
| `mol_graph_direct_useScores.py` | Stage 2 | 候选产物枚举 + 图构建 | **直接复用** |
| `edit_mol_direct_useScores.py` | Stage 2 | 分子编辑（键变化 → 产物） | **直接复用** |
| `models.py` (rank) | Stage 2 | 差异 WLN (wl_diff_net) | **PyTorch 重写** |
| `directcandranker.py` | Stage 2 | CandRanker 完整模型 | **PyTorch 重写** |
| `prep_data.py` | 预处理 | 反应中心提取 (get_changed_bonds) | **直接复用** |

## 5. 两阶段流水线概览

rexgen_direct 将正向反应预测分解为两个子问题：

```
输入：反应物 SMILES（带原子映射编号）

┌─────────────────────────────────────────────────────────────┐
│  Stage 1: CoreFinder（反应中心预测）                         │
│                                                             │
│  反应物 SMILES                                              │
│    ↓                                                        │
│  原子特征 + 键特征 → 分子图                                  │
│    ↓                                                        │
│  WLN 消息传递（3层）→ 原子嵌入 h_i                           │
│    ↓                                                        │
│  全局注意力：h_i + h_j + 二值特征 → 注意力分数               │
│    ↓                                                        │
│  上下文增强的原子对评分 → Top-K 键变化预测                    │
│    ↓                                                        │
│  输出：[(atom_i, atom_j, bond_order, score), ...]           │
└─────────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────────┐
│  候选枚举（Candidate Enumeration）                          │
│                                                             │
│  从 Top-K 键变化中组合生成所有合法候选产物                    │
│  过滤条件：化学价合法、键变化连通、不重复                     │
└─────────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────────┐
│  Stage 2: CandRanker（候选产物排序）                         │
│                                                             │
│  反应物图 + 各候选产物图                                     │
│    ↓                                                        │
│  WLN 编码所有分子 → 候选 - 反应物 = 差异表示                 │
│    ↓                                                        │
│  差异 WLN（1层）→ 反应指纹                                   │
│    ↓                                                        │
│  线性评分 + CoreFinder 偏置 → 候选排序                       │
│    ↓                                                        │
│  输出：排序后的产物 SMILES 列表                               │
└─────────────────────────────────────────────────────────────┘
```

## 6. 核心概念：键变化（Bond Changes）

rexgen_direct 将化学反应建模为**键的变化**。对于反应物和产物之间的每一对原子 $(i, j)$，键的变化用三元组表示：

$$\text{edit} = (\text{atom}_i, \text{atom}_j, \text{bond\_order})$$

其中 `bond_order` 表示产物中该键的阶数：

| bond_order | 含义 | 说明 |
|------------|------|------|
| 0.0 | 键断裂 | 反应物中有键，产物中无键 |
| 1.0 | 单键 | 产物中为单键（可能是新生成或键序变化） |
| 2.0 | 双键 | 产物中为双键 |
| 3.0 | 三键 | 产物中为三键 |
| 1.5 | 芳香键 | 产物中为芳香键 |

这5类变化对应模型输出的5维分类。

下面用一个具体例子演示键变化的提取。

In [4]:
import rdkit.Chem as Chem
import numpy as np

# ====== 源码映射 ======
# 文件: rexgen_direct/scripts/prep_data.py
# 函数: get_changed_bonds()

def get_changed_bonds(rxn_smi):
    """提取反应中发生变化的化学键。
    
    比较反应物和产物中所有原子对之间的键序，找出差异。
    返回: set of (atom_map_1, atom_map_2, new_bond_order)
    """
    reactants = Chem.MolFromSmiles(rxn_smi.split('>')[0])
    products  = Chem.MolFromSmiles(rxn_smi.split('>')[2])

    # 只关注产物中存在的原子（通过 molAtomMapNumber 追踪）
    conserved_maps = [a.GetProp('molAtomMapNumber') 
                      for a in products.GetAtoms() 
                      if a.HasProp('molAtomMapNumber')]
    bond_changes = set()

    # 收集反应物中的键
    bonds_prev = {}
    for bond in reactants.GetBonds():
        nums = sorted([
            bond.GetBeginAtom().GetProp('molAtomMapNumber'),
            bond.GetEndAtom().GetProp('molAtomMapNumber')])
        if (nums[0] not in conserved_maps) and (nums[1] not in conserved_maps):
            continue
        bonds_prev['{}~{}'.format(nums[0], nums[1])] = bond.GetBondTypeAsDouble()

    # 收集产物中的键
    bonds_new = {}
    for bond in products.GetBonds():
        nums = sorted([
            bond.GetBeginAtom().GetProp('molAtomMapNumber'),
            bond.GetEndAtom().GetProp('molAtomMapNumber')])
        bonds_new['{}~{}'.format(nums[0], nums[1])] = bond.GetBondTypeAsDouble()

    # 比较差异
    for bond in bonds_prev:
        if bond not in bonds_new:
            bond_changes.add((bond.split('~')[0], bond.split('~')[1], 0.0))  # 键断裂
        else:
            if bonds_prev[bond] != bonds_new[bond]:
                bond_changes.add((bond.split('~')[0], bond.split('~')[1], bonds_new[bond]))  # 键序变化
    for bond in bonds_new:
        if bond not in bonds_prev:
            bond_changes.add((bond.split('~')[0], bond.split('~')[1], bonds_new[bond]))  # 新键生成

    return bond_changes

In [5]:
# ====== 示例：亲核取代反应 ======
# 甲胺取代芳环上的氯
rxn_smi = (
    '[CH3:14][NH2:15].'
    '[N+:1](=[O:2])([O-:3])[c:4]1[cH:5][c:6]([C:7](=[O:8])[OH:9])[cH:10][cH:11][c:12]1[Cl:13].'
    '[OH2:16]'
    '>>'
    '[N+:1](=[O:2])([O-:3])[c:4]1[cH:5][c:6]([C:7](=[O:8])[OH:9])[cH:10][cH:11][c:12]1[NH:15][CH3:14]'
)

bond_changes = get_changed_bonds(rxn_smi)

print('反应 SMILES（简化显示）:')
print(f'  反应物: {rxn_smi.split(">>")[0][:60]}...')
print(f'  产  物: {rxn_smi.split(">>")[1][:60]}...')
print()
print('键变化:')
print('-' * 50)
for a1, a2, bo in sorted(bond_changes):
    if bo == 0.0:
        change_type = '键断裂'
    elif bo == 1.0:
        change_type = '形成单键'
    elif bo == 2.0:
        change_type = '形成双键'
    elif bo == 1.5:
        change_type = '形成芳香键'
    else:
        change_type = f'键序={bo}'
    print(f'  原子 {a1} - 原子 {a2}: bond_order = {bo}  ({change_type})')

# 与数据文件中的 edits 格式对比
edits_str = ';'.join([f'{a1}-{a2}-{bo}' for a1, a2, bo in sorted(bond_changes)])
print(f'\n编辑字符串: {edits_str}')
print(f'预期值:     12-13-0.0;12-15-1.0')

反应 SMILES（简化显示）:
  反应物: [CH3:14][NH2:15].[N+:1](=[O:2])([O-:3])[c:4]1[cH:5][c:6]([C:...
  产  物: [N+:1](=[O:2])([O-:3])[c:4]1[cH:5][c:6]([C:7](=[O:8])[OH:9])...

键变化:
--------------------------------------------------
  原子 12 - 原子 13: bond_order = 0.0  (键断裂)
  原子 12 - 原子 15: bond_order = 1.0  (形成单键)

编辑字符串: 12-13-0.0;12-15-1.0
预期值:     12-13-0.0;12-15-1.0


## 7. 验证示例数据

教学用的示例数据包含 6 个小分子反应，涵盖不同类型的键变化。

In [7]:
import pandas as pd
from pathlib import Path

# 加载示例数据
demo_path = Path('data/demo_reactions.csv')
if not demo_path.exists():
    # 尝试从 notebook 所在目录加载
    notebook_dir = Path(os.path.dirname(os.path.abspath('__file__')))
    demo_path = notebook_dir / 'data' / 'demo_reactions.csv'

df = pd.read_csv(demo_path)
print(f'示例数据: {len(df)} 条反应')
print(f'列名: {list(df.columns)}')
print()

for idx, row in df.iterrows():
    rxn = row['rxn_smiles']
    edits = row['edits']
    name = row['reaction_name']
    
    # 统计键变化数量和类型
    edit_list = edits.split(';')
    n_edits = len(edit_list)
    
    reactant_smi = rxn.split('>>')[0]
    mol = Chem.MolFromSmiles(reactant_smi)
    n_atoms = mol.GetNumAtoms() if mol else '?'
    
    print(f'反应 {idx+1}: {name}')
    print(f'  原子数: {n_atoms}, 键变化数: {n_edits}')
    print(f'  编辑: {edits}')
    print()

示例数据: 6 条反应
列名: ['rxn_smiles', 'edits', 'reaction_name']

反应 1: 亲核取代_Cl被NH2取代
  原子数: 16, 键变化数: 2
  编辑: 12-13-0.0;12-15-1.0

反应 2: 还原反应_醛还原为醇
  原子数: 16, 键变化数: 1
  编辑: 10-9-1.0

反应 3: 酯化反应_酰氯与酚
  原子数: 19, 键变化数: 2
  编辑: 10-2-0.0;19-2-1.0

反应 4: 酰胺化_胺与酰氯
  原子数: 32, 键变化数: 2
  编辑: 30-31-0.0;3-30-1.0

反应 5: 亲核取代_OTf被NH取代
  原子数: 37, 键变化数: 2
  编辑: 19-3-0.0;3-37-1.0

反应 6: 脲形成_胺与异氰酸酯
  原子数: 23, 键变化数: 2
  编辑: 21-22-1.0;1-22-1.0



In [8]:
# 验证: 对每条反应提取键变化，与标注一致
print('验证键变化提取:')
print('=' * 60)
for idx, row in df.iterrows():
    rxn = row['rxn_smiles']
    expected_edits = row['edits']
    
    extracted = get_changed_bonds(rxn)
    extracted_str = ';'.join(sorted([f'{a1}-{a2}-{bo}' for a1, a2, bo in extracted]))
    expected_set = set()
    for e in expected_edits.split(';'):
        a1, a2, bo = e.split('-')
        expected_set.add((a1, a2, float(bo)))
    
    match = (extracted == expected_set)
    flag = '✓' if match else '✗'
    print(f'  {flag} 反应 {idx+1} ({row["reaction_name"]})')
    if not match:
        print(f'    提取: {extracted}')
        print(f'    预期: {expected_set}')

验证键变化提取:
  ✓ 反应 1 (亲核取代_Cl被NH2取代)
  ✓ 反应 2 (还原反应_醛还原为醇)
  ✓ 反应 3 (酯化反应_酰氯与酚)
  ✓ 反应 4 (酰胺化_胺与酰氯)
  ✓ 反应 5 (亲核取代_OTf被NH取代)
  ✓ 反应 6 (脲形成_胺与异氰酸酯)


## 8. 总结

本 Notebook 完成了以下工作：

1. 安装了 rexgen_direct 教程所需的所有 Python 依赖（PyTorch、RDKit、NumPy）
2. 验证了环境配置正确
3. 概览了源码目录结构和各模块职责
4. 展示了两阶段流水线的整体架构
5. 演示了核心概念——键变化的提取
6. 验证了示例数据文件

### 教学版 vs 原版的核心区别

| 方面 | 原版 | 教学版 |
|------|------|--------|
| 深度学习框架 | TensorFlow 1.x | **PyTorch 2.0+** |
| 数据处理代码 | 原始 RDKit/NumPy | **直接复用**（无需修改） |
| 模型权重 | 有预训练 checkpoint | **随机初始化**（仅演示架构） |
| 图框架 | 自定义邻接表 | **保持自定义**（无需 DGL/PyG） |

**下一步** → 请打开 `2_数据处理.ipynb`，学习完整的数据预处理流程。